# M05 · Statistical seismology: b-value, completeness, aftershock decay

**Week 2 · about 9 h (+ 2 h optional)** · before this: M01, M02, M04

> **한국어 요약** 규모별 지진 수(구텐베르크-리히터 법칙)에서 b값과 그 불확실성, 완전성 규모(Mc)를 계산하는 함수를 직접 만듭니다. 규모 차이로 구하는 b-positive와 여진이 줄어드는 속도(오모리-우츠 법칙)도 구현한 뒤, 2016년 경주 지진의 여진에 적용합니다.

**You will**

- Make practice magnitudes and estimate b with its uncertainty
- Find the magnitude of completeness and see why it matters
- Fit the decay of aftershocks by maximum likelihood
- Report b, b-positive and p for the 2016 Gyeongju aftershocks

**New words:** Gutenberg-Richter law (구텐베르크-리히터 법칙), b-value (b값), magnitude of completeness (Mc; 완전성 규모), maximum likelihood estimation (최대가능도 추정, 최대우도 추정), standard error (표준 오차), b-positive (양의 규모 차이 b값), Omori-Utsu law (오모리-우츠 법칙), declustering (디클러스터링, 종속 지진 제거) · [all terms](../../../docs/glossary.md)

**Optional reading:** Mignan & Woessner (2012), maximum curvature and b-value stability; Utsu, Ogata & Matsu'ura (1995), the modified Omori formula

Log any AI help in `work/ai_log/M05.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M05.yaml](../../../.sgtl/criteria/M05.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
SEED = 2016                    # random seed of the practice catalogs
DM = 0.1                       # magnitude bin width (KMA lists ML with one decimal)
MC_CORRECTION = 0.2            # added to the maximum-curvature Mc (Part 3)
N_REPEAT = 500                 # practice catalogs per Monte Carlo test
SEQUENCE = "gyeongju2016"      # the sequence of task 6
GJ_DMC = 0.2                   # task 6: smallest magnitude difference for b-positive
GJ_T_START_DAYS = 0.1          # task 6: start of the Omori-Utsu fit (days after the mainshock)
GJ_T_END_DAYS = 365.0          # task 6: end of the fit (days after the mainshock)
NATIONAL_YEARS = (2018, 2026)  # task 7: first and last UTC year
NATIONAL_MC = 2.1              # task 7: completeness magnitude
FIGURE_DIR = "figures"         # your figures go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M05")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm

Path(FIGURE_DIR).mkdir(exist_ok=True)
ML = st.plot.ML_LABEL                 # the magnitude label of the house style
rng = np.random.default_rng(SEED)     # one generator for all demos in this notebook

## Part 1 · Counting earthquakes by size (about 60 min)

Count the earthquakes in each magnitude bin and plot the counts on a log axis. Above some magnitude, the points
fall on a straight line, the **Gutenberg-Richter law**:

$$\log_{10} N(\ge M) = a - b\,M$$

$N(\ge M)$ is the number of events of magnitude $M$ or larger. The **b-value** is the slope: with $b = 1$, each
step of one magnitude has ten times fewer events. Most regions have $b$ between 0.8 and 1.2.

Run the cell. Where does the KMA catalog follow a straight line, and where not?

In [ ]:
catalog = st.data.kma_catalog()
since_2018 = catalog[catalog["origin_utc"] >= pd.Timestamp("2018-01-01", tz="UTC")]
per_bin = since_2018["ml"].round(1).value_counts().sort_index()          # events in each 0.1 bin
n_at_or_above = per_bin.iloc[::-1].cumsum().iloc[::-1]                     # events at or above each bin
print(f"{len(since_2018)} events since 2018 | source: {catalog.attrs['source']}")
print(per_bin.loc[1.5:2.3].to_string())

fig, ax = plt.subplots(figsize=(5.2, 3.8), layout="constrained")
ax.semilogy(per_bin.index, per_bin.to_numpy(), "o", ms=4, color="tab:blue", label="Per bin")
ax.semilogy(n_at_or_above.index, n_at_or_above.to_numpy(), "s", ms=4, mfc="none", color="0.25",
            label="At or above")
ax.set(xlabel=f"Local magnitude, {ML}", ylabel="Number of events", title="KMA catalog since 2018 (UTC)")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.1), ncol=2, frameon=False)
plt.show()

Above about ML 2 the counts follow a line. Below about ML 1 they drop: the network misses small events. Just
below ML 2.0 there is a step (M04): the bins 1.8 and 1.9 hold several times more events than 2.0 and 2.1.
Part 3 comes back to it.

**Two facts for task 1.**

- Above a threshold, magnitudes behave like random draws from an **exponential distribution** (each step up is
  less likely by the same factor): $P(M \ge m) = e^{-\beta (m - M_c)}$ with $\beta = b \ln 10$.
- Catalogs **round** magnitudes. KMA lists ML with one decimal, so the **bin width** is $\Delta m = 0.1$: a
  listed 2.3 stands for any magnitude from 2.25 to 2.35.

### M05-T1 · Practice magnitudes from the Gutenberg-Richter law · core · about 60 min

Write `simulate_gr(n, b, mc, dm, rng)`. It returns `n` magnitudes that follow the law with slope `b` above
`mc`, rounded to steps of `dm`.

- `rng.exponential(scale, size)` gives exponential draws whose mean is `scale`; here the mean is $1/\beta$.
- Make continuous magnitudes first, then round them yourself. Take every random number from `rng`, a numpy
  random **Generator** (the same seed gives the same numbers).

Not allowed: SeismoStats, `sgtl_tutorial`, or drawing whole bin numbers (`rng.geometric`, `rng.integers`).

In [ ]:
# Before you code: where must the continuous draws start so that the bin at mc gets its full share?
st.predict("M05-T1", """

""")

In [ ]:
def simulate_gr(n, b, mc, dm, rng):
    """Gutenberg-Richter magnitudes rounded to steps of dm.

    n   : int, how many magnitudes
    b   : float, b-value (base-10 slope)
    mc  : float, smallest rounded magnitude (a whole number of bins)
    dm  : float, bin width
    rng : numpy Generator; every random draw comes from it
    returns : numpy array of n floats, each mc + k*dm with k = 0, 1, 2, ...
    """
    # YOUR CODE HERE
    raise NotImplementedError("M05-T1: write your code here")

In [ ]:
st.check("M05-T1", func=simulate_gr)

**Explain:** Where do your continuous draws start, and why there?

Use it: 100,000 practice magnitudes with $b = 1$ above $M_c = 1.0$. The counts at or above each magnitude should
fall by a factor of ten per magnitude unit.

In [ ]:
practice = simulate_gr(100_000, 1.0, 1.0, DM, rng)
values, counts = np.unique(practice, return_counts=True)
above_counts = counts[::-1].cumsum()[::-1]
print("Ratio of events at or above M and at or above M + 1:",
      np.round(above_counts[:5] / above_counts[10:15], 2))

fig, ax = plt.subplots(figsize=(5.2, 3.8), layout="constrained")
ax.semilogy(values, counts, "o", ms=4, color="tab:blue", label="Per bin")
ax.semilogy(values, above_counts, "s", ms=4, mfc="none", color="0.25", label="At or above")
ax.set(xlabel="Magnitude", ylabel="Number of events", title="Practice catalog, b = 1")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.1), ncol=2, frameon=False)
plt.show()

## Part 2 · Estimating b (about 60 min)

Do not fit a straight line through the counts: the few large events make such a fit noisy. Use **maximum
likelihood** instead: choose the $b$ that makes the observed magnitudes most probable. For magnitudes rounded to
bins of width $\Delta m$, the Aki-Utsu formula is

$$\hat b = \frac{\log_{10} e}{\bar M - (M_c - \Delta m/2)}$$

$\bar M$ is the mean magnitude of the events at or above $M_c$. The $-\Delta m/2$ is there because the lowest bin
starts half a bin below its listed value.

How sure is $\hat b$? Shi and Bolt's formula gives its **standard error**, the typical scatter of $\hat b$ if you
could repeat the experiment:

$$\sigma_b = \ln 10 \cdot b^2 \sqrt{\frac{\sum_i (M_i - \bar M)^2}{n\,(n-1)}}$$

For many events it is close to $b/\sqrt{n}$: about 0.1 for 100 events.

### M05-T2 · b-value and its uncertainty · core · about 90 min

Write `b_utsu(mags, mc, dm)`, the b-value of the events at or above `mc`, and `shi_bolt(mags, b)`, its
standard error.

- `mags` may hold events below `mc`: leave them out. Stored decimals are not exact (2.0 can be
  1.9999999999999998), so keep an event when `m >= mc - dm/2`.
- Use $\ln 10$ in `shi_bolt` (the check also accepts the 2.30 of the paper).

Example: 1.0, 1.0, 1.1, 1.3 with `mc` = 1.0 and `dm` = 0.1 give $\hat b \approx 2.90$.

Not allowed: SeismoStats or `sgtl_tutorial` inside your functions.

In [ ]:
# Before you code: at b = 1 and dm = 0.1, by how many percent will b be off without the half-bin correction?
st.predict("M05-T2", """

""")

In [ ]:
def b_utsu(mags, mc, dm):
    """Corrected Aki-Utsu maximum-likelihood b-value of rounded magnitudes.

    mags : numpy array of magnitudes (a whole number of bins; may include events below mc)
    mc   : float, completeness magnitude (a whole number of bins)
    dm   : float, bin width
    returns : float, b-value (base 10) of the events at or above mc
    """
    # YOUR CODE HERE
    raise NotImplementedError("M05-T2: write your code here")


def shi_bolt(mags, b):
    """Shi and Bolt (1982) uncertainty of a b-value.

    mags : numpy array of the magnitudes used for b (all at or above mc)
    b    : float, their b-value
    returns : float, one standard error of b
    """
    # YOUR CODE HERE
    raise NotImplementedError("M05-T2: write your code here")

In [ ]:
st.check("M05-T2", func=b_utsu, shi_bolt=shi_bolt)

**Explain:** Run the cell below first. Is the error without the half-bin correction larger or smaller than the
scatter of $\hat b$ for 1,000 events?

**Monte Carlo test:** repeat the estimate on `N_REPEAT` random practice catalogs of each size and see how the
results scatter. The last line compares you with SeismoStats, a statistics package from ETH Zurich.

In [ ]:
true_b = 1.0
rows = []
for n_events in (50, 200, 1000):
    est, sig, plain = [], [], []
    for _ in range(N_REPEAT):
        mags = simulate_gr(n_events, true_b, 1.0, DM, rng)
        b_hat = b_utsu(mags, 1.0, DM)
        est.append(b_hat)
        sig.append(shi_bolt(mags, b_hat))
        plain.append(np.log10(np.e) / (mags.mean() - 1.0))     # no half-bin correction, on purpose
    rows.append({"events": n_events, "mean b": np.mean(est), "scatter of b (std)": np.std(est, ddof=1),
                 "mean Shi-Bolt sigma": np.mean(sig), "b over sqrt(n)": true_b / np.sqrt(n_events),
                 "mean b without correction": np.mean(plain)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# The exact binned estimator in SeismoStats, on one big practice catalog:
from seismostats.analysis import ClassicBValueEstimator, UtsuBValueEstimator

big = simulate_gr(200_000, true_b, 1.0, DM, rng)
b_mine = b_utsu(big, 1.0, DM)
b_exact = ClassicBValueEstimator().calculate(big, mc=1.0, delta_m=DM)
print(f"yours {b_mine:.4f} | SeismoStats Utsu {UtsuBValueEstimator().calculate(big, mc=1.0, delta_m=DM):.4f} | "
      f"SeismoStats Classic (Tinti-Mulargia) {b_exact:.4f} | difference {100 * (b_mine / b_exact - 1):+.2f} %")

## Part 3 · The magnitude of completeness (about 60 min)

The formula assumes that no event at or above $M_c$ is missing. The **magnitude of completeness** $M_c$ is the
smallest magnitude for which that holds. Set it too low and $\hat b$ comes out too low; set it too high and you
throw data away.

The simplest estimate is **maximum curvature**: the bin with the most events. Detection usually fades gradually
below $M_c$, so 0.2 is added (`MC_CORRECTION`).

### M05-T3 · Completeness from the peak of the histogram · core · about 60 min

Write `mc_maxc(mags, dm, correction=0.2)`: the center of the bin with the most events, plus `correction`.

- Bins have width `dm` and are centered on the listed values (multiples of `dm`).
- If several bins share the largest count, take the smallest magnitude.
- `np.unique(..., return_counts=True)` counts how often each value occurs.

Example: 1.0, 1.1, 1.1, 1.2 with `dm` = 0.1 give 1.1 + 0.2 = 1.3.

Not allowed: SeismoStats or `sgtl_tutorial` inside your function.

In [ ]:
# Before you code: for a catalog whose detection fades slowly over a whole magnitude unit, will MAXC + 0.2 be
# above or below the magnitude where detection is really complete?
st.predict("M05-T3", """

""")

In [ ]:
def mc_maxc(mags, dm, correction=0.2):
    """Maximum-curvature magnitude of completeness (Wiemer and Wyss 2000) plus a correction.

    mags       : numpy array of rounded magnitudes
    dm         : float, bin width
    correction : float, added to the peak bin (0.2 after Woessner and Wiemer 2005)
    returns    : float, magnitude of completeness
    """
    # YOUR CODE HERE
    raise NotImplementedError("M05-T3: write your code here")

In [ ]:
st.check("M05-T3", func=mc_maxc)

**Explain:** In the demo below, where is MAXC + 0.2 compared with 99 % detection, and what does that do to
$\hat b$?

**Demo: detection that fades gradually.** A practice catalog with $b = 1$. An event of magnitude $m$ is detected
with probability `norm.cdf((m - 1.6) / 0.5)`: 50 % at 1.6, 99 % at about 2.8. SeismoStats' b-stability method
is shown for comparison.

In [ ]:
import contextlib
import io
import warnings

from seismostats.analysis import estimate_mc_b_stability, estimate_mc_maxc
# note: estimate_mc_maxc takes fmd_bin= (its docstring example passes delta_m=, which fails)

DETECT_50, DETECT_WIDTH = 1.6, 0.5        # detection: 50 % at 1.6, a smooth step of this width
full = simulate_gr(150_000, 1.0, 0.0, DM, rng)
seen = full[rng.random(full.size) < norm.cdf((full - DETECT_50) / DETECT_WIDTH)]
mc_peak = mc_maxc(seen, DM, MC_CORRECTION)
with warnings.catch_warnings(), contextlib.redirect_stdout(io.StringIO()):   # keep the library quiet
    warnings.simplefilter("ignore")
    mc_stable, _ = estimate_mc_b_stability(seen, delta_m=DM)
print(f"{seen.size} detected events | yours {mc_peak:.1f} | SeismoStats MAXC "
      f"{estimate_mc_maxc(seen, fmd_bin=DM, correction_factor=MC_CORRECTION)[0]:.1f} | b-stability {mc_stable:.1f}")
print(f"b at MAXC + 0.2: {b_utsu(seen, mc_peak, DM):.2f} | b at the b-stability Mc: {b_utsu(seen, mc_stable, DM):.2f}"
      f" | 99 % detection at about {DETECT_50 + DETECT_WIDTH * norm.ppf(0.99):.1f}")

trial_mc = np.round(np.arange(1.0, 3.21, DM), 1)
trial_b = [b_utsu(seen, m0, DM) for m0 in trial_mc]
trial_s = [shi_bolt(seen[seen >= m0 - DM / 2], bb) for m0, bb in zip(trial_mc, trial_b)]
fig, ax = plt.subplots(figsize=(5.2, 3.8), layout="constrained")
ax.errorbar(trial_mc, trial_b, yerr=trial_s, fmt="o", ms=4, color="tab:blue", label="b at each trial Mc")
ax.axhline(1.0, color="0.4", ls="--", lw=1, label="True b")
ax.axvline(mc_peak, color="tab:red", ls=":", lw=1.2, label="MAXC + 0.2")
ax.axvline(mc_stable, color="tab:green", ls="-.", lw=1.2, label="b-stability")
ax.set(xlabel="Trial completeness magnitude", ylabel="b-value", title="Gradual detection")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.1), frameon=False, ncol=2, fontsize=9)
plt.show()

**Watch out: two traps in the KMA catalog.**

1. The network grew. Before 2008 almost only events of ML 2.0 or more are listed, so estimate $M_c$ for each
   network epoch.
2. Since 2018 the bins 1.8 and 1.9 hold several times the events of 2.0 and 2.1. Never fit one b-value across
   ML 2.0; for a national b-value use $M_c \ge 2.1$.

In [ ]:
epochs = {"1978-2007": ("1978-01-01", "2008-01-01"), "2008-2015": ("2008-01-01", "2016-01-01"),
          "2016-2026": ("2016-01-01", "2027-01-01")}
for name, (t0, t1) in epochs.items():
    part = catalog[(catalog["origin_utc"] >= pd.Timestamp(t0, tz="UTC"))
                   & (catalog["origin_utc"] < pd.Timestamp(t1, tz="UTC"))]["ml"].to_numpy()
    print(f"{name}: {part.size:5d} events, MAXC + {MC_CORRECTION} = {mc_maxc(part, DM, MC_CORRECTION):.1f}")

recent = since_2018["ml"].to_numpy()
for m0 in (1.5, 1.8, 2.1, 2.3):
    b0 = b_utsu(recent, m0, DM)
    used = recent[recent >= m0 - DM / 2]
    print(f"since 2018, Mc {m0:.1f}: {used.size:5d} events, b = {b0:.2f} +- {shi_bolt(used, b0):.2f}")

**Optional: explorer E9.** Sliders for $M_c$ and the bin width redraw the counts with your own `fmd` (from M04),
`b_utsu`, `shi_bolt` and `mc_maxc`. Paste your `fmd` into the cell first. Try $M_c$ from 1.5 to 2.5.

In [ ]:
# Optional explorer E9 (not checked). Paste your fmd function from M04 above this comment to see the counts.
your_fmd = globals().get("fmd")                # your fmd when this notebook has one, otherwise None
mags_to_explore = recent                       # or seen, the practice catalog of the gradual-detection demo
st.widgets.fmd_explorer(mags_to_explore, fmd=your_fmd, b_estimator=b_utsu, sigma_estimator=shi_bolt,
                        mc_estimator=mc_maxc, dm=DM)

## Part 4 · Aftershocks and b-positive (about 60 min)

Right after a large **mainshock**, the network misses many small **aftershocks**: their waves are hidden in the
long shaking of bigger events. So $M_c$ is high for hours to days.

**b-positive** works around this. Take the magnitudes in time order and the difference to the event before,
$d_i = m_{i+1} - m_i$. Keep only differences of at least $\delta_c$ (`dmc`, one bin or more). They follow the
same $b$ as the magnitudes, because an event larger than the one before it is usually recorded. For rounded
magnitudes:

$$\hat b_+ = \frac{1}{\Delta m \ln 10}\,\ln\!\left(1 + \frac{\Delta m}{\bar d - \delta_c}\right)$$

$\bar d$ is the mean of the kept differences.

### M05-T4 · b from magnitude differences · core · about 75 min

Write `b_positive(mags, dm, dmc=None)` with the formula above. `mags` are in time order (earliest first);
`dmc=None` means one bin (`dm`).

- Keep a difference when `d >= dmc - dm/2` (stored decimals are not exact).

Example: 1.0, 1.2, 1.1, 1.4 with `dm` = 0.1 keep the differences 0.2 and 0.3 and give $\hat b_+ \approx 2.22$.

Not allowed: SeismoStats or `sgtl_tutorial` inside your function.

In [ ]:
# Before you code: in the first day after a mainshock, will the classic b at the long-term Mc be too high or
# too low?
st.predict("M05-T4", """

""")

In [ ]:
def b_positive(mags, dm, dmc=None):
    """b-positive (van der Elst 2021) from the positive magnitude differences of a time-ordered catalog.

    mags : numpy array of rounded magnitudes in time order (earliest first)
    dm   : float, bin width
    dmc  : float or None, smallest difference used (a whole number of bins; None means dm)
    returns : float, b-value (base 10)
    """
    # Step 1: decide the threshold, then take the differences of neighbors in time.
    # Step 2: keep the differences at or above the threshold (with the half-bin margin of the task text).
    # Step 3: put the mean of the kept differences into the formula, with dmc in the place of Mc.
    # YOUR CODE HERE
    raise NotImplementedError("M05-T4: write your code here")

In [ ]:
st.check("M05-T4", func=b_positive)

**Explain:** In the demo below, which estimate is closer to the true $b = 1$ in the first day, and why?

**Demo: an aftershock sequence with missing small events.** 20,000 practice aftershocks ($b = 1$ above 1.0)
between 0.001 and 100 days. An aftershock is missed when it is smaller than
$M_c(t) = 6.0 - 4.5 - 0.75\log_{10} t$ ($t$ in days).

In [ ]:
n_after = 20_000
t_days = 10 ** rng.uniform(-3.0, 2.0, n_after)
order = np.argsort(t_days)
t_days = t_days[order]
m_true = simulate_gr(n_after, 1.0, 1.0, DM, rng)
mc_t = 6.0 - 4.5 - 0.75 * np.log10(t_days)
listed = m_true >= mc_t
print(f"{listed.sum()} of {n_after} aftershocks are listed")

for label, window in (("first day", t_days <= 1.0), ("days 10-100", t_days >= 10.0)):
    sub = m_true[listed & window]
    print(f"{label:12s}: {sub.size:5d} listed | classic b at Mc 1.0: {b_utsu(sub, 1.0, DM):.2f} | "
          f"b-positive (dmc 0.2): {b_positive(sub, DM, 0.2):.2f}")

# The price of b-positive: scatter in complete practice catalogs of 500 events.
classic, positive = [], []
for _ in range(N_REPEAT):
    mags = simulate_gr(500, 1.0, 1.0, DM, rng)
    classic.append(b_utsu(mags, 1.0, DM))
    positive.append(b_positive(mags, DM, 0.2))
print(f"500 complete events: std of classic b {np.std(classic, ddof=1):.3f}, "
      f"std of b-positive {np.std(positive, ddof=1):.3f}")

## Part 5 · Aftershock decay: the Omori-Utsu law (about 90 min)

The aftershock rate falls with the time $t$ (days) after the mainshock:

$$\lambda(t) = \frac{K}{(t + c)^p}$$

- $K$: how many aftershocks there are.
- $p$ (usually 0.9-1.3): how fast the rate falls.
- $c$ (minutes to hours): keeps the rate finite at $t = 0$.

To fit it, use the log-likelihood of the event times $t_1, \dots, t_n$ seen between $t_s$ and $t_e$:

$$\ln L = \sum_{i=1}^{n} \ln \lambda(t_i) - \int_{t_s}^{t_e} \lambda(t)\,dt$$

The sum rewards a high rate where events happened. The integral, the number of events the model expects, keeps
the rate from growing everywhere.

The integral has a closed form:

$$\int_{t_s}^{t_e} \frac{K\,dt}{(t+c)^p} = K\,\frac{(t_e+c)^{1-p} - (t_s+c)^{1-p}}{1-p}\ \ (p \ne 1),
\qquad K \ln\frac{t_e + c}{t_s + c}\ \ (p = 1)$$

`scipy.optimize.minimize(f, x0, method="Nelder-Mead")` searches for the minimum of $-\ln L$. Let it work on
$\ln K$ and $\ln c$, so they stay positive, and start it from several values of $c$.

### M05-T5 · Fitting the aftershock decay · core · about 120 min

Write `omori_negloglik(params, t, t_start, t_end)`: it returns $-\ln L$ for `params` = (K, c, p). Then write
`fit_omori(t, t_start, t_end)`: it returns the (K, c, p) that minimize it.

- Times are in days. K, c and p are plain numbers, not logarithms.
- `omori_negloglik` must also work for $p = 1$ exactly.

Example: `omori_negloglik((10, 1, 1), np.array([1.0]), 0, 1)` gives $10 \ln 2 - \ln 5 \approx 5.32$.

Not allowed: `sgtl_tutorial` inside your functions.

In [ ]:
# Before you code: if you start the fit at t_start = 1 day instead of 0, which of K, c, p will become the most
uncertain?
st.predict("M05-T5", """

""")

In [ ]:
def omori_negloglik(params, t, t_start, t_end):
    """Negative log-likelihood of the Omori-Utsu rate K/(t + c)**p for event times t in [t_start, t_end].

    params : (K, c, p) with K in events per day, c in days, p without unit
    t      : numpy array of event times, days after the mainshock
    t_start, t_end : float, the observation window in days
    returns : float, minus the log-likelihood
    """
    # Step 1: the sum of the log-rates at the event times (log K minus p times log(t + c), for every event).
    # Step 2: the expected number of events in the window (the integral of Part 5; p = 1 has its own form).
    # Step 3: minus the log-likelihood = expected number minus the sum of Step 1.
    # YOUR CODE HERE
    raise NotImplementedError("M05-T5: write your code here")


def fit_omori(t, t_start, t_end):
    """Maximum-likelihood Omori-Utsu parameters (K, c, p) of event times t (days) in [t_start, t_end]."""
    # Step 1: write a small function of the free parameters (for example log c and p) that returns the
    #         negative log-likelihood; get K from the other two or let the optimizer handle log K as well.
    # Step 2: run scipy.optimize.minimize from several starting values of c and keep the lowest result.
    # Step 3: turn the winner back into plain K, c, p.
    # YOUR CODE HERE
    raise NotImplementedError("M05-T5: write your code here")

In [ ]:
st.check("M05-T5", func=omori_negloglik, fit=fit_omori)

**Explain:** In the demos below, which of $c$ and $p$ changes most when the fit starts later, and how does the
contour plot explain it?

**Demo: practice sequences and the c-p trade-off.** `omori_times` draws event times that follow the law. The
contour plot shows $-\ln L$ around the best fit: a long, tilted valley means the data cannot tell $c$ and $p$
apart well.

In [ ]:
def omori_times(K, c, p, t0, t1, rng):
    """Practice event times (days) of an Omori-Utsu sequence on [t0, t1], by inverse transform sampling."""
    expected = K * (np.log((t1 + c) / (t0 + c)) if p == 1 else
                    ((t1 + c) ** (1 - p) - (t0 + c) ** (1 - p)) / (1 - p))
    u = rng.random(rng.poisson(expected))
    if p == 1:
        return np.sort((t0 + c) * ((t1 + c) / (t0 + c)) ** u - c)
    lo, hi = (t0 + c) ** (1 - p), (t1 + c) ** (1 - p)
    return np.sort((lo + u * (hi - lo)) ** (1 / (1 - p)) - c)


K_TRUE, C_TRUE, P_TRUE = 300.0, 0.05, 1.1
times_all = omori_times(K_TRUE, C_TRUE, P_TRUE, 0.0, 365.0, rng)
print(f"{times_all.size} practice aftershocks; true K = {K_TRUE}, c = {C_TRUE} days, p = {P_TRUE}")
for t_s in (0.01, 0.1, 1.0):
    k_f, c_f, p_f = fit_omori(times_all[times_all >= t_s], t_s, 365.0)
    print(f"start {t_s:5.2f} days: K = {k_f:7.1f}, c = {c_f:.4f} days, p = {p_f:.3f}")

t_s = 0.1
sub = times_all[times_all >= t_s]
log_c_grid = np.linspace(np.log(0.001), np.log(2.0), 60)
p_grid = np.linspace(0.9, 1.4, 60)
surface = np.empty((p_grid.size, log_c_grid.size))
for i, p_try in enumerate(p_grid):
    for j, lc in enumerate(log_c_grid):
        expect_k1 = omori_negloglik((1.0, np.exp(lc), p_try), np.empty(0), t_s, 365.0)   # no events: the integral
        surface[i, j] = omori_negloglik((sub.size / expect_k1, np.exp(lc), p_try), sub, t_s, 365.0)
fig, ax = plt.subplots(figsize=(5.2, 3.8), layout="constrained")
levels = surface.min() + np.array([0.5, 2.0, 4.5, 8.0, 12.5])          # 1 to 5 "sigma" steps
ax.contour(np.exp(log_c_grid), p_grid, surface, levels=levels, colors="tab:blue")
ax.plot(C_TRUE, P_TRUE, "*", ms=12, color="tab:red", label="True (c, p)")
ax.set(xscale="log", xlabel="c (days)", ylabel="p", title=f"Fit from {t_s} days: likelihood valley")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.1), ncol=1, frameon=False)
plt.show()

**Demo: plotting a rate.** Count the events in bins evenly spaced in log time and divide each count by the bin
width in days. Plot each rate at the bin center on a log axis, $\sqrt{t_\mathrm{left}\,t_\mathrm{right}}$.

In [ ]:
bin_edges = np.geomspace(0.001, 365.0, 26)                  # 25 bins from 0.001 to 365 days
per_bin, _ = np.histogram(times_all, bins=bin_edges)        # events in each bin
width_days = np.diff(bin_edges)
centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])           # the middle of each bin on a log axis
filled = per_bin > 0                                         # an empty bin has no point on a log axis
t_line = np.geomspace(0.001, 365.0, 200)
print(f"{per_bin.sum()} of {times_all.size} events in the bins | first bin: {per_bin[0]} events in "
      f"{width_days[0] * 1440:.1f} minutes | last bin: {per_bin[-1]} events in {width_days[-1]:.0f} days")

fig, ax = plt.subplots(figsize=(5.2, 3.8), layout="constrained")
ax.loglog(centers[filled], per_bin[filled] / width_days[filled], "o", ms=4, color="tab:blue",
          label="Count / bin width")
ax.loglog(t_line, K_TRUE / (t_line + C_TRUE) ** P_TRUE, "-", color="tab:red", label="True rate $K/(t+c)^p$")
ax.set(xlabel="Days since the mainshock", ylabel="Events per day", title="Practice sequence: aftershock rate")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.1), ncol=2, frameon=False)
plt.show()

## Part 6 · The 2016 Gyeongju aftershocks (about 120 min)

On 12 September 2016 an ML 5.8 earthquake struck near Gyeongju, the largest instrumentally recorded earthquake in
South Korea. The sequence `gyeongju2016` holds the catalog events within 15 km of it, from 2016-09-01 to
2017-09-30. In task 6 you choose its $M_c$ yourself. First look at the counts near ML 2.0: do you see the step of
Part 1?

In [ ]:
gyeongju = st.data.sequence(SEQUENCE)
print(gyeongju.attrs["sequence"]["description"], "|", len(gyeongju), "events |", gyeongju.attrs["source"])
print(gyeongju["ml"].round(1).value_counts().sort_index().loc[1.4:2.6].to_string())

### M05-T6 · The 2016 Gyeongju aftershocks · core · about 120 min

Write `aftershock_summary(seq, dm, mc, dmc, t_start, t_end)`. It finds the mainshock and the later events, then
uses **your** functions of tasks 2-5. It returns a dict with Mc, $b \pm \sigma_b$, b-positive, the Omori-Utsu
fit and a two-panel figure. The docstring lists every key.

Use only the arguments: the check also runs your function on a made-up sequence. Then set `GJ_MC` from your own
numbers. Not allowed: SeismoStats.

In [ ]:
# Before you code: what Mc, b and p do you expect for the Gyeongju aftershocks?
st.predict("M05-T6", """

""")

In [ ]:
def aftershock_summary(seq, dm, mc, dmc, t_start, t_end):
    """Mc, b-values and Omori-Utsu fit of the aftershocks in seq.

    seq     : DataFrame like st.data.sequence(...) (event_id, origin_utc in UTC, ml, ...)
    dm, mc  : float, magnitude bin width and completeness magnitude
    dmc     : float, smallest magnitude difference for b-positive
    t_start, t_end : float, window of the Omori-Utsu fit, days after the mainshock
    returns : dict with
        events      the aftershocks: rows of seq after the mainshock (largest ml, earliest if tied), in time order
        mc_maxc     your mc_maxc of their magnitudes (dm, default correction)
        mc, t_start, t_end  the values you were given
        b, sigma_b  your b_utsu and shi_bolt of the magnitudes at or above mc
        b_positive  your b_positive of those at or above mc (time order, dmc)
        K, c, p     your fit_omori of their times (days after the mainshock), at or above mc and inside the window
        figure      a Figure: (a) counts per bin and at or above (log scale), mc marked, your b-line;
                    (b) the fitted events' counts in bins evenly spaced in log time, divided by the bin widths
                    (days), at the bin centers, and your fitted rate as a line; both axes log
    """
    # YOUR CODE HERE
    raise NotImplementedError("M05-T6: write your code here")

**Choose your $M_c$.** The cell runs your function with a first guess (your MAXC of all events) and prints
$\hat b$ for trial values of $M_c$. Then set `GJ_MC` in the cell after it.

In [ ]:
first_guess = mc_maxc(gyeongju["ml"].to_numpy(), DM)          # a first guess only
look = aftershock_summary(gyeongju, DM, first_guess, GJ_DMC, GJ_T_START_DAYS, GJ_T_END_DAYS)
after_ml = look["events"]["ml"].to_numpy()
print(f"{after_ml.size} aftershocks | first guess {first_guess:.1f} | your mc_maxc of the aftershocks "
      f"{look['mc_maxc']:.1f}")
for m0 in np.round(np.arange(1.3, 2.81, DM), 1):
    used = after_ml[after_ml >= m0 - DM / 2]
    b0 = b_utsu(after_ml, m0, DM)
    print(f"trial Mc {m0:.1f}: {used.size:5d} events, b = {b0:.2f} +- {shi_bolt(used, b0):.2f}")
plt.show()

In [ ]:
# Set GJ_MC, your completeness magnitude for the Gyeongju aftershocks: one value, a whole number of bins.
# The check needs at least 50 aftershocks at or above it, and tests that their magnitudes follow the
# Gutenberg-Richter law with your b.
# YOUR CODE HERE
raise NotImplementedError("M05-T6: write your code here")

In [ ]:
st.check("M05-T6", func=aftershock_summary, mc=GJ_MC, dmc=GJ_DMC, t_start=GJ_T_START_DAYS,
         t_end=GJ_T_END_DAYS)

**Explain:** Why did you choose your `GJ_MC`? Use the counts and the trial table above.

Your numbers for Gyeongju, the figure, and how $c$ and $p$ move with the start of the fit:

In [ ]:
summary = aftershock_summary(gyeongju, DM, GJ_MC, GJ_DMC, GJ_T_START_DAYS, GJ_T_END_DAYS)
used_n = int((summary["events"]["ml"].to_numpy() >= GJ_MC - DM / 2).sum())
print(f"{len(summary['events'])} aftershocks; MAXC + 0.2 = {summary['mc_maxc']:.1f}; Mc used {summary['mc']:.1f} "
      f"({used_n} events)")
print(f"b = {summary['b']:.2f} +- {summary['sigma_b']:.2f} | b-positive (dmc {GJ_DMC}) = {summary['b_positive']:.2f}")
print(f"Omori-Utsu from {GJ_T_START_DAYS} to {GJ_T_END_DAYS} days: K = {summary['K']:.1f} per day, "
      f"c = {summary['c']:.4f} days, p = {summary['p']:.2f}")
for ext in ("png", "pdf"):
    summary["figure"].savefig(Path(FIGURE_DIR) / f"gyeongju2016_aftershocks.{ext}")
plt.show()

aftershock_mags = summary["events"]["ml"].to_numpy()
days_after = ((summary["events"]["origin_utc"] - gyeongju.loc[gyeongju["ml"].idxmax(), "origin_utc"])
              .dt.total_seconds().to_numpy() / 86400.0)
for t_s in (0.01, 0.1, 1.0):
    pick = (aftershock_mags >= GJ_MC - DM / 2) & (days_after >= t_s) & (days_after <= GJ_T_END_DAYS)
    k_f, c_f, p_f = fit_omori(days_after[pick], t_s, GJ_T_END_DAYS)
    print(f"start {t_s:4.2f} days: {pick.sum():4d} events, c = {c_f:.4f} days, p = {p_f:.2f}")
print("Source:", gyeongju.attrs["source"])

## Part 7 · Optional: declustering the national catalog (about 90 min)

**Declustering** removes aftershocks and foreshocks to keep the **background seismicity**: the events not
triggered by earlier ones. Gardner and Knopoff's recipe puts a space window and a time window, both growing with
magnitude, around every event; smaller events inside both are removed. SeismoStats implements it.

**Watch out:** SeismoStats drops time zones silently. Convert times to UTC without a zone first, and never mix
UTC and KST.

In [ ]:
from seismostats.analysis import GardnerKnopoffType1, GardnerKnopoffWindow

window = GardnerKnopoffWindow()
space_km, time_windows = window(np.array([2.5, 3.5, 4.5, 5.5]))
time_days = np.array([w / pd.Timedelta(days=1) for w in time_windows])   # SeismoStats gives Timedelta objects
print(pd.DataFrame({"ML": [2.5, 3.5, 4.5, 5.5], "space window (km)": np.round(space_km, 1),
                    "time window (days)": np.round(time_days, 1)}).to_string(index=False))

### M05-T7 · Declustering the national catalog · stretch · about 90 min

Write `decluster_national(cat, years, mc, dm)`. It selects the events of the given UTC years at or above `mc`
and declusters them with `GardnerKnopoffType1(GardnerKnopoffWindow())`. It returns a dict with the events, the
kept flags, $b$ before and after, and a figure of both distributions. The docstring lists every key.

In [ ]:
# Before you code: will declustering raise or lower the national b-value, and by more than its uncertainty?
st.predict("M05-T7", """

""")

In [ ]:
def decluster_national(cat, years, mc, dm):
    """Gardner-Knopoff declustering (SeismoStats) of the events of the given UTC years at or above mc.

    cat   : DataFrame like st.data.kma_catalog()
    years : (first, last) UTC years, both included
    mc    : float, completeness magnitude (for the KMA catalog since 2018: 2.1 or more, never across ML 2.0)
    dm    : float, magnitude bin width
    returns : dict with
        events           the selected rows (UTC years, ml >= mc - dm/2) in time order, index 0, 1, 2, ...
        mainshock_flags  numpy bool array, one per row of events: True where GardnerKnopoffType1 keeps the event
        mc               the mc you were given
        b_all, b_declustered  b at mc of all selected events and of the kept ones (Aki-Utsu or exact binned)
        figure           a matplotlib Figure with both frequency-magnitude distributions
    GardnerKnopoffType1 takes a table with the columns time (UTC without a zone), magnitude, latitude and longitude,
    and the index 0, 1, 2, ...
    """
    # YOUR CODE HERE
    raise NotImplementedError("M05-T7: write your code here")

In [ ]:
st.check("M05-T7", func=decluster_national, mc=NATIONAL_MC)

**Explain:** How many events did declustering remove, and did $b$ change by more than its uncertainty?

In [ ]:
national = decluster_national(catalog, NATIONAL_YEARS, NATIONAL_MC, DM)
flags = national["mainshock_flags"]
print(f"{len(national['events'])} events of ML {NATIONAL_MC} or more; {flags.sum()} kept, {(~flags).sum()} removed")
print(f"b all {national['b_all']:.2f} +- {shi_bolt(national['events']['ml'].to_numpy(), national['b_all']):.2f} | "
      f"b declustered {national['b_declustered']:.2f}")
years_utc = national["events"]["origin_utc"].dt.year
print(pd.DataFrame({"all": years_utc.value_counts().sort_index(),
                    "kept": years_utc[flags].value_counts().sort_index()}).fillna(0).astype(int).T.to_string())
for ext in ("png", "pdf"):
    national["figure"].savefig(Path(FIGURE_DIR) / f"national_declustering.{ext}")
plt.show()

## Optional: research ideas (not checked)

- **Nearest-neighbor clustering** gives each event a "parent": the earlier event with the smallest
  $\eta = t\,r^{1.6}\,10^{-b\,m}$ (time, distance, magnitude). Two humps in the histogram of $\log_{10}\eta$
  (cell below) mean clustered and background events.
- **ETAS** lets every event trigger its own Omori-Utsu sequence. Try fitting it to Gyeongju.

In [ ]:
# Optional: nearest-neighbor proximity for the events of task 7 (takes a few seconds).
from pyproj import Geod

ev = national["events"]
years_since = (ev["origin_utc"] - ev["origin_utc"].iloc[0]).dt.total_seconds().to_numpy() / (365.25 * 86400.0)
lat, lon, mag = (ev[k].to_numpy(dtype=float) for k in ("latitude", "longitude", "ml"))
geod = Geod(ellps="WGS84")
log_eta = np.full(len(ev), np.nan)
for j in range(1, len(ev)):
    _, _, metres = geod.inv(np.full(j, lon[j]), np.full(j, lat[j]), lon[:j], lat[:j])
    r_km = np.maximum(metres / 1000.0, 0.1)              # 0.1 km floor: catalog locations are rounded
    gap = np.maximum(years_since[j] - years_since[:j], 1e-6)
    log_eta[j] = np.min(np.log10(gap) + 1.6 * np.log10(r_km) - 1.0 * mag[:j])
fig, ax = plt.subplots(figsize=(5.2, 3.8), layout="constrained")
ax.hist(log_eta[np.isfinite(log_eta)], bins=40, color="tab:blue")
ax.set(xlabel="log10 of the nearest-neighbor proximity", ylabel="Number of events",
       title="Clustered (left) and background (right)")
plt.show()

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M05")

**Optional:**

1. Repeat task 6 for `pohang2017`.
2. Run the Monte Carlo test of Part 2 for b-positive with `dmc` 0.1, 0.2 and 0.3.
3. Set `DETECT_WIDTH` to 0.1 and 0.3 in the gradual-detection demo.
4. Compare your $b$ and $p$ of task 6 with Kim and Lee (2019), in Korean.

**Read more:** Mignan and Woessner (2012), completeness and b; van der Elst (2021), b-positive; Utsu et al.
(1995), the Omori-Utsu law. The [reading list](../../../docs/reading_list.md) has the rest.

Next: M03 (getting seismic data), then M10 (seismograms). Commit your notebook as in M00.